In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from numba import jit
import scipy
import pandas as pd

In [3]:
from integrator import Particles
from integrator import Particle
from integrator import constants
from integrator import DataIO
from integrator import Simulation
from integrator import WH_SA_P
from integrator import WH_SB_P
from integrator import WH_SC_P
from integrator import WH_SAB_P
from integrator import WH_SABC_P

In [5]:
planets = pd.read_csv('../output/planets_triples.csv')

In [15]:
planets.head()

,Unnamed: 0,pl_name,pl_orbper,pl_orbsmax,pl_rade,pl_bmasse,pl_orbeccen,pl_eqt,sy_dist
0,0,16 Cyg B b,798.500000,1.6600,13.50,565.737400,0.680,NaN,21.13970
1,1,51 Eri b,9100.000000,10.4000,NaN,3464.329636,0.570,807.0,29.75750
2,2,91 Aqr b,181.400000,0.7000,13.10,1017.000000,0.027,NaN,44.03040
3,3,BD-14 3065 b,4.288973,0.0656,21.59,3932.000000,0.066,2001.0,589.42300
4,4,GJ 229 A c,121.932680,0.3840,2.87,8.581367,0.366,NaN,5.75624


In [6]:
triples = pd.read_csv('../input/triple_systems_updated.csv')

In [7]:
triples.head()

,system_name,catalog_name,system_type,host_star,mA_Msun,mB_Msun,mC_Msun,RA_Rsun,RB_Rsun,RC_Rsun,...,outer_P_yr,outer_e,outer_i_deg,outer_Omega_deg,outer_omega_deg,outer_sep_arcsec,outer_T0,source_masses,source_radii,source_orbits
0,Gliese 667,HIP 84709 / HD 156384,S(C),C,0.730,0.69,0.33,0.760,0.7,0.42,...,2523.0,NaN,NaN,NaN,NaN,32.420,NaN,MSC,Stellar catalog,MSC
1,Alpha Centauri,HIP 71683 / HD 128620,S(C),C,1.100,0.88,0.12,1.220,0.93,0.154,...,547000.0,0.50,107.60,126.00,72.30,7920.000,2850.0,"MSC, https://alphacen2023.sciencesconf.org/dat...",Stellar catalog,"MSC, https://alphacen2023.sciencesconf.org/dat..."
2,LTT 1445,HIP 14101,S(A),A,0.257,0.215,0.161,0.271,0.236,0.197,...,NaN,NaN,2.88,NaN,NaN,7.281,NaN,https://en.wikipedia.org/wiki/LTT_1445,https://en.wikipedia.org/wiki/LTT_1445,https://en.wikipedia.org/wiki/LTT_1445 Evans/Z...
3,94 Ceti,HD 19994,S(A),A,1.340,0.55,0.34,1.221,0.4,0.36,...,1420.0,0.26,114.10,84.13,247.74,6.770,1982.0,MSC; NASA archive (A),Pecaut & Mamajek,MSC VB6_Hle1994; VB6_Sef2010a
4,51 Eri,HD 29391,S(A),A,1.750,0.65,0.45,1.473,0.5,0.42,...,55144.0,NaN,NaN,NaN,NaN,66.960,NaN,MSC,Pecaut & Mamajek,MSC VB6_Mtt2015


## Gliese 667

In [16]:
GJ667 = triples.iloc[0]
GJ667_pl = planets.loc[planets["pl_name"].str.contains("GJ 667", na=False)]

In [19]:
GJ667_pl

,Unnamed: 0,pl_name,pl_orbper,pl_orbsmax,pl_rade,pl_bmasse,pl_orbeccen,pl_eqt,sy_dist
6,6,GJ 667 C b,7.203,0.050431,2.25,5.68,0.20,NaN,7.24396
7,7,GJ 667 C c,28.140,0.125000,1.77,3.80,0.02,NaN,7.24396
8,8,GJ 667 C e,62.240,0.213000,1.45,2.70,0.02,NaN,7.24396
9,9,GJ 667 C f,39.026,0.156000,1.45,2.70,0.03,NaN,7.24396
10,10,GJ 667 C g,256.200,0.549000,1.99,4.60,0.08,NaN,7.24396


In [ ]:
particles = Particles(constants.G, system_type='S(C)-P')

# Star A: placed at origin
particles.add_particle(Particle(
    ptype=0, mass=GJ667['mA_Msun'], radius=GJ667['RA_Rsun'], name='star A',
    temperature=GJ667['TA_K'],
    pos=[0.0, 0.0, 0.0], vel=[0.0, 0.0, 0.0],
    index='A'
))

# Star B: inner binary companion — orbits star A
particles.add_particle(Particle(
    ptype=0, mass=GJ667['mB_Msun'], radius=GJ667['RB_Rsun'], name='star B', temperature=GJ667['TB_K'],
    a=GJ667['inner_a_AU'], e=GJ667['inner_e'], inc=GJ667['inner_i_deg'], Omega=GJ667['inner_Omega_deg'], 
    omega=GJ667['inner_omega_deg'], theta=GJ667['inner_T0'],
    angles_in_degrees=True, index='B', primary='star A'
))

# Star C: outer companion — orbits COM(A+B)
particles.add_particle(Particle(
    ptype=0, mass=GJ667['mC_Msun'], radius=GJ667['RC_Rsun'], name='star C', temperature=GJ667['TC_K'],
    a=GJ667['outer_a_AU'], e=GJ667['outer_e'], inc=GJ667['outer_i_deg'], Omega=GJ667['outer_Omega_deg'], 
    omega=GJ667['outer_omega_deg'], theta=GJ667['outer_T0'],
    angles_in_degrees=True, index='c', primary=['star A', 'star B']
))

for i in len(GJ667_pl):
    # Planet: S-type around C — orbits star C
    particles.add_particle(Particle(
        ptype=1, mass=GJ667_pl['pl_bmasse'][i]*3.00274e-6, radius=GJ667_pl['pl_rade'][i]*0.00916794, name=f'planet {i}',
        a=1.0, e=0.05, inc=0.0, Omega=0.0, omega=0.0, theta=0.0,
        angles_in_degrees=False, index='P', primary='star C'
    ))

# Shift to the system COM frame
m = particles.masses
r_com = (m[:, None] * particles.pos).sum(axis=0) / m.sum()
v_com = (m[:, None] * particles.vel).sum(axis=0) / m.sum()
particles._pos -= r_com[None, :]
particles._vel -= v_com[None, :]
particles._sync_objects()

print(particles._particles)
print("star_indices:", particles.star_indices)
print("planet_indices:", particles.planet_indices)
print(f"COM position: {(particles.masses[:, None] * particles.pos).sum(axis=0) / particles.masses.sum()}")
print(f"COM velocity: {(particles.masses[:, None] * particles.vel).sum(axis=0) / particles.masses.sum()}")

In [ ]:
from integrator import run_system

# Build configs from your DataFrames
configs = []
for _, row in triples.iterrows():
    pl_subset = planets[planets['pl_name'].str.contains(row['system_name'])]
    configs.append({
        'system_type': row['system_type'],
        'host_star':   row['host_star'],
        'mA': row['mA_Msun'], 'mB': row['mB_Msun'], 'mC': row['mC_Msun'],
        'RA': row['RA_Rsun'], 'RB': row['RB_Rsun'], 'RC': row['RC_Rsun'],
        'TA': row['TA_K'],    'TB': row['TB_K'],    'TC': row['TC_K'],
        'inner_a': row['inner_a_AU'], 'inner_e': row['inner_e'],
        'inner_i': row['inner_i_deg'], 'inner_Omega': row['inner_Omega_deg'],
        'inner_omega': row['inner_omega_deg'], 'inner_T0': row['inner_T0'],
        'outer_a': row['outer_a_AU'], 'outer_e': row['outer_e'],
        'outer_i': row['outer_i_deg'], 'outer_Omega': row['outer_Omega_deg'],
        'outer_omega': row['outer_omega_deg'], 'outer_T0': row['outer_T0'],
        'planets': [{'mass_earth': p['pl_bmasse'], 'radius_earth': p['pl_rade'],
                      'a': p['pl_orbsmax'], 'e': p['pl_orbeccen']}
                     for _, p in pl_subset.iterrows()],
        'tf': 1e4, 'dt': 0.001, 'output_every_n': 10,
        'output_file': f'/tmp/sim_{row["system_name"]}.hdf5',
    })

# Run in parallel on Spark
results = sc.parallelize(configs).map(run_system).collect()